In [1]:
import numpy as np
import math

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# WEEK 13 — FUNCTION 2 (RL-AWARE GP-BASED LOCAL MAXIMISATION)
# What changed vs Week 12 (Module 24: RL lens):
#  - Treats candidate generators as "actions" in a Multi-Armed Bandit (MAB)
#    arms = {global, local(PCA), PC-line, jitter(PCA)}
#  - Runs a short probe phase: samples each arm, measures reward proxy (best EI)
#  - Updates per-arm Q-values (Q-learning-style incremental update)
#  - Allocates the final candidate budget using a softmax policy + epsilon exploration
#    (exploration–exploitation balance becomes data-adaptive as n grows)
#  - Adds a lightweight Thompson-sampling term (model-free stochastic exploration)
#  - Keeps PCA novelty + PC-guidance from Week 12 to avoid redundant sampling
#  - Output x_next is rounded to <= 6 decimals
# ============================================================

# ----------------------------
# 1) INPUT DATA (your history)
# ----------------------------
X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],   # will be clipped to 1.0
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],   # duplicate
    [0.99947700, 0.02152800],
    [0.00757800, 0.97735900],
    [1.00000000, 0.98231600],
    [0.69158300, 0.48913100],
    [0.68265500, 0.34268700],
    [0.71120900, 0.66530500]
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439,
    0.15170334833240093, 0.003163976344064868, 0.4846853396945425,
    0.6052619631635735, 0.6292692117412668
], dtype=float)

# Bounds enforcement for black-box
X = np.clip(X, 0.0, 1.0)

# -------------------------------------------
# 2) DEDUPE (average y for identical points)
# -------------------------------------------
def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)
    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())
    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# -------------------------------------------
# 3) GP model (interpretable uncertainty)
# -------------------------------------------
def make_gp(seed=2026):
    d = X.shape[1]
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
        length_scale=np.ones(d),
        length_scale_bounds=(1e-2, 1e1),
        nu=2.5
    ) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=12,
        random_state=seed
    )

def gp_cv_mse(X, y, seed=2026):
    kf = KFold(n_splits=min(5, len(y)), shuffle=True, random_state=seed)
    mses = []
    for tr, te in kf.split(X):
        gp = make_gp(seed=seed)
        gp.fit(X[tr], y[tr])
        mu, _ = gp.predict(X[te], return_std=True)
        mses.append(mean_squared_error(y[te], mu))
    return float(np.mean(mses))

cv_mse = gp_cv_mse(X, y, seed=2026)

gp = make_gp(seed=2026)
gp.fit(X, y)

# -------------------------------------------
# 4) Acquisition helpers
# -------------------------------------------
def normal_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)

def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))

def expected_improvement(mu, std, y_best, xi=0.001):
    std = np.maximum(std, 1e-12)
    z = (mu - y_best - xi) / std
    return (mu - y_best - xi) * normal_cdf(z) + std * normal_pdf(z)

def zscore(v):
    return (v - v.mean()) / (v.std() + 1e-12)

# -------------------------------------------
# 5) PCA lens (keep Week 12 structure)
# -------------------------------------------
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=2026)
Xp = pca.fit_transform(Xz)
evr = pca.explained_variance_ratio_

# PC most correlated with y (directional guidance)
corrs = []
for j in range(2):
    v = Xp[:, j]
    if np.std(v) < 1e-12:
        corrs.append(0.0)
    else:
        corrs.append(float(np.corrcoef(v, y)[0, 1]))
pc_star = int(np.argmax(np.abs(corrs)))
pc_star_sign = 1.0 if corrs[pc_star] >= 0 else -1.0

# -------------------------------------------
# 6) Setup: best-so-far & trust region scale
# -------------------------------------------
rng = np.random.default_rng(2026)

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])
n = len(y)

x_best_p = pca.transform(scaler.transform(x_best.reshape(1, -1))).reshape(-1)

# Slightly tighter than Week 12 (final round), but still allows escape
base = float(np.clip(0.90 / np.sqrt(max(n, 1)), 0.14, 0.30))
pc_scales = np.array([
    base * np.sqrt(max(evr[0], 1e-6)) / np.sqrt(max(evr.mean(), 1e-6)),
    base * np.sqrt(max(evr[1], 1e-6)) / np.sqrt(max(evr.mean(), 1e-6)),
], dtype=float)
pc_scales = np.clip(pc_scales, 0.06, 0.34)

def pca_to_raw(Xp_samples):
    Xz_samples = pca.inverse_transform(Xp_samples)
    Xraw = scaler.inverse_transform(Xz_samples)
    return np.clip(Xraw, 0.0, 1.0)

def novelty_p(Xcand):
    Xcand_p = pca.transform(scaler.transform(Xcand))
    dists_p = np.sqrt(((Xcand_p[:, None, :] - Xp[None, :, :]) ** 2).sum(axis=2))
    min_dist_p = dists_p.min(axis=1)
    return min_dist_p, Xcand_p

# Separation threshold in PCA space
min_sep_p = float(np.clip(0.85 / np.sqrt(max(n, 1)), 0.10, 0.20))

# -------------------------------------------
# 7) RL/MAB: define "actions" (candidate generators)
# -------------------------------------------
arms = ["global", "local_p", "line", "jitter_p"]

def sample_candidates(arm, m):
    if arm == "global":
        return rng.uniform(0.0, 1.0, size=(m, 2))

    if arm == "local_p":
        Xp_local = rng.normal(loc=x_best_p, scale=pc_scales, size=(m, 2))
        return pca_to_raw(Xp_local)

    if arm == "line":
        t = rng.normal(loc=0.0, scale=1.0, size=(m, 1))
        line_dir = np.zeros((1, 2))
        line_dir[0, pc_star] = 1.0
        perp_dir = np.zeros((1, 2))
        perp_dir[0, 1 - pc_star] = 1.0

        Xp_line = x_best_p + (t * (pc_scales[pc_star] * 2.4)) * line_dir
        Xp_line = Xp_line + rng.normal(
            0.0, pc_scales[1 - pc_star] * 0.40, size=(m, 1)
        ) * perp_dir
        return pca_to_raw(Xp_line)

    if arm == "jitter_p":
        q = np.quantile(y, 0.75)
        top_idx = np.where(y >= q)[0]
        center = Xp[top_idx].mean(axis=0) if len(top_idx) > 0 else x_best_p
        Xp_jitter = rng.normal(loc=center, scale=pc_scales * 0.85, size=(m, 2))
        return pca_to_raw(Xp_jitter)

    raise ValueError("Unknown arm")

# -------------------------------------------
# 8) RL/MAB probe phase (feedback-driven Q update)
#    Reward proxy = best EI from that arm (valid points)
# -------------------------------------------
Q = np.zeros(len(arms), dtype=float)
N = np.zeros(len(arms), dtype=int)

# epsilon decays with n: more exploitation as dataset grows
epsilon = float(np.clip(0.35 / np.sqrt(max(n, 1)), 0.08, 0.18))

# Q-learning style fast update (within this final decision)
alpha = 0.6

probe_per_arm = 2500

for i, arm in enumerate(arms):
    Xc = sample_candidates(arm, probe_per_arm)
    mu_c, std_c = gp.predict(Xc, return_std=True)

    # Slightly larger xi when uncertain, but shrinks with n
    xi = float(np.clip(0.002 / np.sqrt(max(n, 1)), 0.0005, 0.002))
    ei_c = expected_improvement(mu_c, std_c, y_best, xi=xi)

    md_c, _ = novelty_p(Xc)
    valid_c = md_c >= min_sep_p

    r = float(np.max(np.where(valid_c, ei_c, -np.inf)))
    if not np.isfinite(r):
        r = float(np.max(ei_c))

    # One-step Q update: Q <- Q + alpha*(r - Q)
    Q[i] = Q[i] + alpha * (r - Q[i])
    N[i] += 1

# Softmax policy over Q (policy optimisation), mixed with epsilon exploration
temp = 0.004  # low temperature => more greedy among arms
logits = (Q - Q.max()) / max(temp, 1e-12)
p = np.exp(logits)
p = p / p.sum()
p_mix = (1.0 - epsilon) * p + epsilon * (np.ones_like(p) / len(p_mix := p))

# -------------------------------------------
# 9) Final pool allocation using learned policy
# -------------------------------------------
TOTAL = 70000

counts = np.maximum(1, (p_mix * TOTAL).astype(int))
diff = TOTAL - counts.sum()
if diff > 0:
    for j in np.argsort(-p_mix):
        counts[j] += 1
        diff -= 1
        if diff == 0:
            break
elif diff < 0:
    for j in np.argsort(p_mix):
        if counts[j] > 1:
            counts[j] -= 1
            diff += 1
            if diff == 0:
                break

pools = []
arm_id_for_row = []
for arm, c in zip(arms, counts):
    Xc = sample_candidates(arm, int(c))
    pools.append(Xc)
    arm_id_for_row.extend([arm] * len(Xc))

Xcand = np.vstack(pools)
arm_id_for_row = np.array(arm_id_for_row, dtype=object)

# -------------------------------------------
# 10) Score candidates: EI + UCB + Thompson + novelty + PC-guidance
# -------------------------------------------
mu, std = gp.predict(Xcand, return_std=True)

xi = float(np.clip(0.0015 / np.sqrt(max(n, 1)), 0.0004, 0.0015))
ei = expected_improvement(mu, std, y_best, xi=xi)

# kappa adapts mildly with n (less frantic exploration late)
kappa = float(np.clip(2.6 - 0.4 * np.tanh((n - 10) / 10), 2.0, 2.8))
ucb = mu + kappa * std

min_dist_p, Xcand_p = novelty_p(Xcand)
valid = min_dist_p >= min_sep_p

# Thompson-like draw (model-free exploration signal)
ts = mu + std * rng.standard_normal(size=len(mu))

# PCA-guidance term
pc_coord = pc_star_sign * Xcand_p[:, pc_star]
pc_guide = zscore(pc_coord)

# Normalize components
ei_z = zscore(ei)
ucb_z = zscore(ucb)
ts_z = zscore(ts)
nov_z = zscore(min_dist_p)

# Final mixture:
# - EI dominates (exploit best-known)
# - UCB + Thompson provide uncertainty-driven exploration
# - novelty prevents redundant sampling
# - PC-guidance nudges in historically improving direction
w_ei, w_ucb, w_ts, w_nov, w_pc = 0.60, 0.18, 0.10, 0.08, 0.04
score = w_ei * ei_z + w_ucb * ucb_z + w_ts * ts_z + w_nov * nov_z + w_pc * pc_guide

score_masked = np.where(valid, score, -np.inf)
best_idx = int(np.argmax(score_masked))

x_next = np.round(Xcand[best_idx], 6)

# -------------------------------------------
# 11) Transparency prints
# -------------------------------------------
topk = 5
top_idx = np.argsort(score_masked)[-topk:][::-1]

print("================================================")
print("WEEK 13 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)")
print("================================================")
print(f"GP kernel learned: {gp.kernel_}")
print(f"CV MSE (sanity check): {cv_mse:.6f}")
print(f"x_best = [{x_best[0]:.6f}, {x_best[1]:.6f}], y_best = {y_best:.6f}")
print("------------------------------------------------")
print("PCA summary (on standardized X):")
print(f"  explained_variance_ratio = [{evr[0]:.6f}, {evr[1]:.6f}]")
print(f"  corr(PC1,y)={corrs[0]:.6f}  corr(PC2,y)={corrs[1]:.6f}")
print(f"  pc_star = PC{pc_star+1} (sign={pc_star_sign:+.0f})")
print(f"  pc_scales = [{pc_scales[0]:.6f}, {pc_scales[1]:.6f}]")
print("------------------------------------------------")
print("RL/MAB (actions=generators):")
print(f"  arms = {arms}")
print(f"  Q-values (after probe) = {np.round(Q, 6).tolist()}")
print(f"  epsilon = {epsilon:.6f}  softmax_p = {np.round(p, 6).tolist()}")
print(f"  mixed_policy_p = {np.round(p_mix, 6).tolist()}")
print(f"  final allocation counts = {counts.tolist()}")
print("------------------------------------------------")
print(f"min_sep_p (PCA-space) = {min_sep_p:.6f}")
print("------------------------------------------------")
print(f"x_next = [{x_next[0]:.6f}, {x_next[1]:.6f}]")
print(f"mu(x_next)          = {mu[best_idx]:.6f}")
print(f"std(x_next)         = {std[best_idx]:.6f}")
print(f"EI(x_next)          = {ei[best_idx]:.6f}")
print(f"UCB(x_next)         = {ucb[best_idx]:.6f}")
print(f"TS(x_next)          = {ts[best_idx]:.6f}")
print(f"min_dist_p(x_next)  = {min_dist_p[best_idx]:.6f}")
print(f"pc_guide(x_next)    = {pc_guide[best_idx]:.6f}")
print(f"score_parts = EI_w={w_ei}, UCB_w={w_ucb}, TS_w={w_ts}, nov_w={w_nov}, pc_w={w_pc}")
print("------------------------------------------------")
print("Top candidates (for transparency):")
for rank, i in enumerate(top_idx, start=1):
    x = Xcand[i]
    print(
        f"{rank}) arm={arm_id_for_row[i]:>8} x=[{x[0]:.6f},{x[1]:.6f}]  "
        f"mu={mu[i]:.6f} std={std[i]:.6f} EI={ei[i]:.6f} "
        f"UCB={ucb[i]:.6f} TS={ts[i]:.6f} minDistP={min_dist_p[i]:.6f} "
        f"pcGuide={pc_guide[i]:.6f} score={score_masked[i]:.6f}"
    )

C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for

WEEK 13 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)
GP kernel learned: 0.847**2 * Matern(length_scale=[0.0182, 10], nu=2.5) + WhiteKernel(noise_level=0.0069)
CV MSE (sanity check): 0.027905
x_best = [0.683406, 0.063769], y_best = 0.629731
------------------------------------------------
PCA summary (on standardized X):
  explained_variance_ratio = [0.515502, 0.484498]
  corr(PC1,y)=0.145803  corr(PC2,y)=-0.109101
  pc_star = PC1 (sign=+1)
  pc_scales = [0.199417, 0.193328]
------------------------------------------------
RL/MAB (actions=generators):
  arms = ['global', 'local_p', 'line', 'jitter_p']
  Q-values (after probe) = [0.013945, 0.015081, 0.010224, 0.014619]
  epsilon = 0.080000  softmax_p = [0.256006, 0.340047, 0.100985, 0.302962]
  mixed_policy_p = [0.255525, 0.332843, 0.112906, 0.298725]
  final allocation counts = [17886, 23300, 7903, 20911]
------------------------------------------------
min_sep_p (PCA-space) = 0.185485
------------------------------------------------
x_n

C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
